In [ ]:
# пример правильных подключений к api gigachat
from langchain_gigachat.chat_models import GigaChat
from langchain_gigachat.embeddings import GigaChatEmbeddings
from langchain_gigachat.chat_models.gigachat import trim_content_to_stop_sequence
from time import perf_counter, sleep
from typing import List, Optional, Any, Union
from langchain_core.messages import BaseMessage
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.outputs import ChatResult
from langchain_core.language_models.chat_models import generate_from_stream


GIGA_DELAY = 6
GIGA_LAST_INVOKE = 0

class GigaChatDelayed(GigaChat):
    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        stream: Optional[bool] = None,
        **kwargs: Any,
    ) -> ChatResult:
        global GIGA_LAST_INVOKE

        # --- Ожидание
        if perf_counter() - GIGA_LAST_INVOKE >= GIGA_DELAY: # Учет задержки Rate Limiter
            pass
        else:
            sleep(GIGA_DELAY - (perf_counter() - GIGA_LAST_INVOKE))

        GIGA_LAST_INVOKE = perf_counter()
        # ---

        should_stream = stream if stream is not None else self.streaming
        if should_stream:
            stream_iter = self._stream(
                messages, stop=stop, run_manager=run_manager, **kwargs
            )
            return generate_from_stream(stream_iter)

        payload = self._build_payload(messages, **kwargs)
        response = self._client.chat(payload)
        for choice in response.choices:
            trimmed_content = trim_content_to_stop_sequence(
                choice.message.content, stop
            )
            if isinstance(trimmed_content, str):
                choice.message.content = trimmed_content
                break

        return self._create_chat_result(response)


MAX_BATCH_SIZE_CHARS = 1000000
MAX_BATCH_SIZE_PARTS = 90


class GigaChatEmbeddingsDelayed(GigaChatEmbeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        result: List[List[float]] = []
        size = 0
        local_texts = []
        embed_kwargs = {}

        global GIGA_LAST_INVOKE

        # --- Ожидание
        if perf_counter() - GIGA_LAST_INVOKE >= GIGA_DELAY: # Учет задержки Rate Limiter
            pass
        else:
            sleep(GIGA_DELAY - (perf_counter() - GIGA_LAST_INVOKE))

        GIGA_LAST_INVOKE = perf_counter()
        # ---

        if self.model is not None:
            embed_kwargs["model"] = self.model

        for text in texts:
            local_texts.append(text)
            size += len(text)
            if size > MAX_BATCH_SIZE_CHARS or len(local_texts) > MAX_BATCH_SIZE_PARTS:
                for embedding in self._client.embeddings(
                    texts=local_texts, **embed_kwargs
                ).data:
                    result.append(embedding.embedding)
                size = 0
                local_texts = []

        if local_texts:
            for embedding in self._client.embeddings(
                texts=local_texts, **embed_kwargs
            ).data:
                result.append(embedding.embedding)

        return result


In [ ]:
# Структурированный вывод с pydantic
import os
from typing import List, Optional
from pydantic import BaseModel, Field


llm = GigaChatDelayed(
base_url=os.getenv('GIGACHAT_API_URL'),
access_token=os.getenv('JPY_API_TOKEN'),
model="GigaChat-2"
)


class PersonInfo(BaseModel):
"""Модель для извлечения информации о человеке."""
name: str = Field(..., description="Имя человека")
age: Optional[int] = Field(None, description="Возраст")
city: Optional[str] = Field(None, description="Город проживания")
interests: List[str] = Field(default_factory=list, description="Список увлечений")


structured_llm = llm.with_structured_output(PersonInfo)

query = "Ивану 35 лет, он живёт в Москве и увлекается горными лыжами, чтением и программированием."

result = structured_llm.invoke(query)

print(result)

In [ ]:
# пример использования gigachat как агента
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
import os
from pprint import pprint


llm = GigaChatDelayed(
base_url=os.getenv('GIGACHAT_API_URL'),
access_token=os.getenv('JPY_API_TOKEN'),
model="GigaChat-2"
)


@tool
def calculator(expression: str) -> str:
    """Вычисляет математическое выражение, например: '2 + 3 * 4'"""
    try:
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names, {})

        return str(result)

    except Exception as e:
        return f"Ошибка вычисления: {e}"


tools = [calculator]


agent_executor = create_react_agent(
model=llm,
tools=tools
)


messages = agent_executor.invoke({'messages': ['Сколько будет 52 умножить на 48?']})['messages']

pprint(messages)
print()
print(messages[-1].content)
